In [1]:
pip install numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2 scipy==1.13.0 matplotlib==3.8.4 seaborn==0.13.2 GEOparse==2.0.4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
Step 1: Download and preprocess the GSE45827 breast cancer gene expression dataset.
Splits into 3 balanced batches with a simulated batch effect.
"""

import GEOparse
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os

DATASET_ID = "GSE45827"
DATA_DIR   = "data"
os.makedirs(DATA_DIR, exist_ok=True)


def download_and_preprocess():
    print(f"Downloading {DATASET_ID} from GEO (this may take a minute)...")
    gse = GEOparse.get_GEO(geo=DATASET_ID, destdir=DATA_DIR, silent=True)

    print("Extracting expression matrix and labels...")
    expr_frames = []
    sample_ids  = []
    labels      = {}

    for gsm_name, gsm in gse.gsms.items():
        if gsm.table is None or gsm.table.empty:
            continue

        col = gsm.table.set_index("ID_REF")["VALUE"]
        expr_frames.append(col)
        sample_ids.append(gsm_name)

        # GSE45827 stores subtype in characteristics_ch1
        # e.g. "tissue: luminal A" or "subtype: HER2+" etc.
        label = "unknown"
        for ch in gsm.metadata.get("characteristics_ch1", []):
            ch_lower = ch.lower()
            if any(kw in ch_lower for kw in ["luminal", "her2", "basal", "normal", "tnbc"]):
                label = ch.split(":")[-1].strip()
                break
            if "tissue" in ch_lower or "subtype" in ch_lower or "type" in ch_lower:
                label = ch.split(":")[-1].strip()
        labels[gsm_name] = label

    expr_df = pd.concat(expr_frames, axis=1)
    expr_df.columns = sample_ids
    expr_df = expr_df.dropna()

    print(f"Raw matrix: {expr_df.shape[0]} probes x {expr_df.shape[1]} samples")

    # Top 300 most variable genes
    top_genes = expr_df.var(axis=1).nlargest(300).index
    expr_df   = expr_df.loc[top_genes]

    # Z-score normalise
    scaler      = StandardScaler()
    expr_scaled = pd.DataFrame(
        scaler.fit_transform(expr_df.T).T,
        index=expr_df.index, columns=expr_df.columns
    )

    label_series = pd.Series(labels, name="subtype").reindex(sample_ids)
    print("Label distribution:", label_series.value_counts().to_dict())

    # Balanced 3-batch split (interleaved so each batch has all subtypes)
    n     = expr_scaled.shape[1]
    idx   = np.arange(n)
    rng   = np.random.default_rng(42)

    splits = [idx[0::3], idx[1::3], idx[2::3]]
    for b_num, b_idx in enumerate(splits, start=1):
        bdf   = expr_scaled.iloc[:, b_idx].copy()
        # Simulated batch effect: additive column-wise shift
        shift = rng.normal(loc=b_num * 0.4, scale=0.05, size=(bdf.shape[0], 1))
        bdf   = bdf + shift
        bdf.to_csv(os.path.join(DATA_DIR, f"batch_{b_num}.csv"))
        print(f"  batch_{b_num}: {bdf.shape[1]} samples")

    label_series.to_csv(os.path.join(DATA_DIR, "true_labels.csv"), header=True)
    print(f"\nAll files saved to '{DATA_DIR}/'")


if __name__ == "__main__":
    download_and_preprocess()

D:\c programming\proj\.venv\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Extracting expression matrix and labels...
Raw matrix: 29873 probes x 155 samples
Label distribution: {'Basal': 41, 'Her2': 30, 'Luminal B': 30, 'Luminal A': 29, 'unknown': 14, 'None (normal)': 11}
  batch_1: 52 samples
  batch_2: 52 samples
  batch_3: 51 samples

All files saved to 'data/'


## Step 2: Consensus Clustering Pipeline

Loads the batch CSVs, runs K-means and spectral ensembles on all samples, fuses them via ADMM, evaluates, and saves plots to .

In [3]:
"""
Distributed Consensus Clustering — Final Fixed Version
"""

import os, warnings
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
DATA_DIR   = "data"          # relative — works on any OS
PLOTS_DIR  = os.path.join("data", "plots")
N_CLUSTERS = 6
N_ITER     = 100
RHO        = 1.0

os.makedirs(PLOTS_DIR, exist_ok=True)


# ──────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────────
def load_batches():
    labels_df      = pd.read_csv(os.path.join(DATA_DIR, "true_labels.csv"), index_col=0)
    all_sample_ids = labels_df.index.tolist()

    batches       = {}
    batch_sid_map = {}

    for fname in sorted(os.listdir(DATA_DIR)):
        if fname.startswith("batch_") and fname.endswith(".csv"):
            key = fname.replace(".csv", "")
            df  = pd.read_csv(os.path.join(DATA_DIR, fname), index_col=0)
            batches[key]       = df.T.values          # (n_samples, n_genes)
            batch_sid_map[key] = df.columns.tolist()
            print(f"Loaded {key}: {batches[key].shape}")

    # Stack in sorted batch order and align true_labels to same order
    ordered_sids = []
    for key in sorted(batches.keys()):
        ordered_sids.extend(batch_sid_map[key])

    X_all       = np.vstack([batches[k] for k in sorted(batches.keys())])
    le          = LabelEncoder()
    raw         = labels_df.loc[ordered_sids].iloc[:, 0].values
    true_labels = le.fit_transform(raw)

    print(f"\nTotal samples : {X_all.shape[0]}, Features: {X_all.shape[1]}")
    print(f"Classes       : {le.classes_}")
    return X_all, true_labels, le.classes_


def run_kmeans(X, k=N_CLUSTERS, seed=42):
    return KMeans(n_clusters=k, n_init=20, random_state=seed).fit_predict(X)


def run_spectral(X, k=N_CLUSTERS, seed=42):
    return SpectralClustering(
        n_clusters=k, affinity="rbf", gamma=0.01,
        random_state=seed, n_init=10
    ).fit_predict(X)


def co_association_matrix(labels: np.ndarray) -> np.ndarray:
    lbl = np.asarray(labels)
    return (lbl[:, None] == lbl[None, :]).astype(np.float64)


def admm_consensus(co_matrices, n_clusters=N_CLUSTERS, n_iter=N_ITER, rho=RHO):
    """
    Minimise  Σ_b ||Z - C_b||_F²   s.t. Z ∈ [0,1]^{n×n}, Z = Zᵀ

    Local z_b / scaled dual u_b formulation:
        z_b ← clip( (C_b + ρ(Z - u_b)) / (1+ρ) , 0, 1 )
        Z   ← clip( symmetrise( mean_b(z_b + u_b) ) , 0, 1 )
        u_b ← u_b + z_b - Z

    Z is initialised as mean(C_b) + small Gaussian noise so that
    iterations are non-trivial even when all C_b are similar.
    """
    B = len(co_matrices)
    n = co_matrices[0].shape[0]
    rng = np.random.default_rng(0)

    # Init: mean + tiny noise to ensure non-trivial first step
    Z = np.clip(np.mean(co_matrices, axis=0)
                + rng.normal(0, 1e-3, (n, n)), 0.0, 1.0)
    Z = (Z + Z.T) / 2.0

    z = [C.copy() for C in co_matrices]
    u = [np.zeros((n, n)) for _ in range(B)]

    residuals = []
    print(f"\n  {'Iter':>5}  {'Residual (Frob)':>18}")
    print(f"  {'-'*5}  {'-'*18}")

    for it in range(n_iter):
        Z_old = Z.copy()

        # z_b update (closed-form minimiser)
        for b in range(B):
            z[b] = np.clip(
                (co_matrices[b] + rho * (Z - u[b])) / (1.0 + rho),
                0.0, 1.0
            )

        # Z update
        Z = np.mean([z[b] + u[b] for b in range(B)], axis=0)
        Z = np.clip((Z + Z.T) / 2.0, 0.0, 1.0)

        # dual update
        for b in range(B):
            u[b] += z[b] - Z

        res = np.linalg.norm(Z - Z_old, "fro")
        residuals.append(res)

        if (it + 1) % 10 == 0 or it < 3:
            print(f"  {it+1:>5}  {res:>18.8f}")

        if res < 1e-6:
            print(f"\n  → Converged at iteration {it + 1}")
            break

    print(f"\n  Total iterations run: {len(residuals)}")
    return Z, residuals


def spectral_on_consensus(Z, k=N_CLUSTERS):
    return SpectralClustering(
        n_clusters=k, affinity="precomputed",
        random_state=42, n_init=20
    ).fit_predict(Z)


def evaluate(name, pred, true, X, Z=None):
    ari = adjusted_rand_score(true, pred)
    nmi = normalized_mutual_info_score(true, pred)
    try:
        if Z is not None:
            D = np.clip(1.0 - Z, 0.0, None)
            np.fill_diagonal(D, 0.0)
            sil = silhouette_score(D, pred, metric="precomputed")
        else:
            sil = silhouette_score(X, pred)
    except Exception:
        sil = float("nan")
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  ARI        (↑ better, max 1) : {ari:.4f}")
    print(f"  NMI        (↑ better, max 1) : {nmi:.4f}")
    print(f"  Silhouette (↑ better)        : {sil:.4f}")
    return {"name": name, "ARI": ari, "NMI": nmi, "Silhouette": sil}


def plot_pca(X, labels_dict):
    pca  = PCA(n_components=2, random_state=42)
    Xp   = pca.fit_transform(X)
    var  = pca.explained_variance_ratio_

    n = len(labels_dict)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4.5))
    if n == 1:
        axes = [axes]

    for ax, (title, lbl) in zip(axes, labels_dict.items()):
        sc = ax.scatter(Xp[:, 0], Xp[:, 1],
                        c=lbl, cmap="tab10", s=30, alpha=0.85)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(f"PC1 ({var[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({var[1]*100:.1f}%)")
        plt.colorbar(sc, ax=ax, label="Cluster")

    plt.suptitle("PCA Projection — All Methods", fontsize=13, y=1.02)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "pca_clusters.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"PCA plot saved → {path}")
    plt.close()


def plot_consensus_matrix(Z, consensus_labels):
    fig, ax = plt.subplots(figsize=(7, 6))
    order = np.argsort(consensus_labels)          # sort by cluster for block view
    Z_vis = Z[np.ix_(order, order)]
    sns.heatmap(Z_vis, cmap="Blues", ax=ax,
                xticklabels=False, yticklabels=False,
                vmin=0, vmax=1,
                cbar_kws={"label": "Co-association score"})
    ax.set_title("ADMM Consensus Co-association Matrix\n"
                 "(samples sorted by consensus cluster)", fontsize=11)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "consensus_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Consensus matrix saved → {path}")
    plt.close()


def plot_admm_residuals(residuals):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    iters = list(range(1, len(residuals) + 1))

    if len(residuals) == 1:
        # Single point: show as scatter with annotation
        ax.scatter(iters, residuals, color="steelblue", s=60, zorder=3)
        ax.annotate(f"Converged on iter 1\nres={residuals[0]:.2e}",
                    xy=(1, residuals[0]), xytext=(1.2, residuals[0] * 1.5),
                    fontsize=9, color="steelblue")
    else:
        ax.plot(iters, residuals, color="steelblue", linewidth=1.8)
        if all(r > 0 for r in residuals):
            ax.set_yscale("log")

    ax.set_xlabel("Iteration")
    ax.set_ylabel("Primal Residual (Frobenius norm)")
    ax.set_title(f"ADMM Convergence  ({len(residuals)} iterations)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "admm_convergence.png")
    plt.savefig(path, dpi=150)
    print(f"ADMM convergence plot saved → {path}")
    plt.close()


def plot_metrics(results):
    df = pd.DataFrame(results).set_index("name")
    ax = df[["ARI", "NMI", "Silhouette"]].plot(
        kind="bar", figsize=(9, 4.5), colormap="Set2", edgecolor="white"
    )
    plt.title("Clustering Method Comparison", fontsize=13)
    plt.ylabel("Score")
    plt.xticks(rotation=20, ha="right")
    plt.ylim(-0.05, 1.1)
    plt.axhline(0, color="gray", linewidth=0.7)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "metrics_comparison.png")
    plt.savefig(path, dpi=150)
    print(f"Metrics plot saved → {path}")
    plt.close()

def main():
    X_all, true_labels, class_names = load_batches()

    # Build ensemble co-association matrices (all on full dataset)
    print("\n--- K-means ensemble (3 random seeds) ---")
    co_mats = []
    km_labels_list = []
    for seed in [42, 7, 123]:
        lbl = run_kmeans(X_all, seed=seed)
        km_labels_list.append(lbl)
        co_mats.append(co_association_matrix(lbl))
        print(f"  seed={seed}  clusters={np.unique(lbl)}")

    print("\n--- Spectral ensemble (2 random seeds) ---")
    sc_labels_list = []
    for seed in [42, 7]:
        lbl = run_spectral(X_all, seed=seed)
        sc_labels_list.append(lbl)
        co_mats.append(co_association_matrix(lbl))
        print(f"  seed={seed}  clusters={np.unique(lbl)}")

    print(f"\nTotal co-association matrices fed to ADMM: {len(co_mats)}")

    # ADMM
    print(f"\n--- ADMM Consensus ({N_ITER} max iterations, rho={RHO}) ---")
    Z_consensus, residuals = admm_consensus(
        co_mats, n_clusters=N_CLUSTERS, n_iter=N_ITER, rho=RHO
    )
    consensus_labels = spectral_on_consensus(Z_consensus, N_CLUSTERS)

    # Baselines
    print("\n--- Baseline: K-means (single run) ---")
    baseline_km = run_kmeans(X_all, seed=42)
    print("\n--- Baseline: Spectral (single run) ---")
    baseline_sc = run_spectral(X_all, seed=42)

    # Evaluate all
    results = [
        evaluate("Baseline K-means",  baseline_km,      true_labels, X_all),
        evaluate("Baseline Spectral", baseline_sc,      true_labels, X_all),
        evaluate("ADMM Consensus",    consensus_labels, true_labels, X_all, Z=Z_consensus),
    ]

    # Plots
    plot_consensus_matrix(Z_consensus, consensus_labels)
    plot_admm_residuals(residuals)
    plot_pca(X_all, {
        "True Labels":      true_labels,
        "Baseline K-means": baseline_km,
        "ADMM Consensus":   consensus_labels,
    })
    plot_metrics(results)

    print("\nDone! All plots saved to data/plots/")


if __name__ == "__main__":
    main()


Loaded batch_1: (52, 300)
Loaded batch_2: (52, 300)
Loaded batch_3: (51, 300)

Total samples : 155, Features: 300
Classes       : ['Basal' 'Her2' 'Luminal A' 'Luminal B' 'None (normal)' 'unknown']

--- K-means ensemble (3 random seeds) ---
  seed=42  clusters=[0 1 2 3 4 5]
  seed=7  clusters=[0 1 2 3 4 5]
  seed=123  clusters=[0 1 2 3 4 5]

--- Spectral ensemble (2 random seeds) ---
  seed=42  clusters=[0 1 2 3 4 5]
  seed=7  clusters=[0 1 2 3 4 5]

Total co-association matrices fed to ADMM: 5

--- ADMM Consensus (100 max iterations, rho=1.0) ---

   Iter     Residual (Frob)
  -----  ------------------
      1          0.04644196
      2          0.02322098
      3          0.01161049
     10          0.00009071

  → Converged at iteration 17

  Total iterations run: 17

--- Baseline: K-means (single run) ---

--- Baseline: Spectral (single run) ---

  Baseline K-means
  ARI        (↑ better, max 1) : 0.6034
  NMI        (↑ better, max 1) : 0.6929
  Silhouette (↑ better)        : 0.169